# 01. 한국 의약품 ID 온톨로지 정리

모든 한국 데이터셋을 연결하는 식별자 체계를 정리한다.

| 식별자 | 설명 | 주로 쓰는 데이터셋 |
|---|---|---|
| **품목일련번호** | 식약처 허가 단위 기본 키 | 낱알식별, e약은요, DUR |
| **표준코드 (바코드)** | 유통 단위 코드 | 처방전, 약국 POS |
| **성분코드** | 주성분 단위 코드 | DUR 병용금기 |
| **ATC 코드** | WHO 분류 체계 | DrugBank, 국제 매핑 |

> **핵심 목표**: `품목일련번호` ↔ `성분코드` ↔ `ATC` 간 crosswalk 테이블 생성

In [16]:
import os, json
import requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

ROOT = Path("../..").resolve()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

DATAGOKR_KEY = os.environ.get("DATAGOKR_NADAL_KEY", "")

In [17]:
# 낱알식별 API — 첫 100건 조회하여 스키마 확인
def fetch_nadal(page=1, rows=100):
    url = "http://apis.data.go.kr/1471000/MdcinGrnIdntfcInfoService03/getMdcinGrnIdntfcInfoList03"
    params = {
        "serviceKey": DATAGOKR_KEY,
        "pageNo": page,
        "numOfRows": rows,
        "type": "json"
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

sample = fetch_nadal()
print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])

{
  "header": {
    "resultCode": "00",
    "resultMsg": "NORMAL SERVICE."
  },
  "body": {
    "pageNo": 1,
    "totalCount": 25518,
    "numOfRows": 100,
    "items": [
      {
        "ITEM_SEQ": "200808876",
        "ITEM_NAME": "가스디알정50밀리그램(디메크로틴산마그네슘)",
        "ENTP_SEQ": "19540006",
        "ENTP_NAME": "일동제약(주)",
        "CHART": "녹색의 원형 필름코팅정",
        "ITEM_IMAGE": "https://nedrug.mfds.go.kr/pbp/cmn/itemImageDownload/147426403087300104",
        "PRINT_FRONT": "IDG",
        "PRINT_BACK": null,
        "DRUG_SHAPE": "원형",
        "COLOR_CLASS1": "연두",
        "COLOR_CLASS2": null,
        "LINE_FRONT": null,
        "LINE_BACK": null,
        "LENG_LONG": "7.6",
        "LENG_SHORT": "7.6",
        "THICK": "3.6",
        "IMG_REGIST_TS": "20100326",
        "CLASS_NO": "02390",
        "CLASS_NAME": "기타의 소화기관용약",
        "ETC_OTC_NAME": "전문의약품",
        "ITEM_PERMIT_DATE": "20080820",
        "FORM_CODE_NAME": "당의정",
        "MARK_CODE_FRONT_ANAL": "",
        "MARK_CODE_BA

In [18]:
# 스키마 및 컬럼 확인
items = sample.get("body", {}).get("items", [])
if items:
    df_sample = pd.DataFrame(items)
    print("컬럼:", df_sample.columns.tolist())
    print("\n샘플 1행:")
    display(df_sample.head(1).T)

컬럼: ['ITEM_SEQ', 'ITEM_NAME', 'ENTP_SEQ', 'ENTP_NAME', 'CHART', 'ITEM_IMAGE', 'PRINT_FRONT', 'PRINT_BACK', 'DRUG_SHAPE', 'COLOR_CLASS1', 'COLOR_CLASS2', 'LINE_FRONT', 'LINE_BACK', 'LENG_LONG', 'LENG_SHORT', 'THICK', 'IMG_REGIST_TS', 'CLASS_NO', 'CLASS_NAME', 'ETC_OTC_NAME', 'ITEM_PERMIT_DATE', 'FORM_CODE_NAME', 'MARK_CODE_FRONT_ANAL', 'MARK_CODE_BACK_ANAL', 'MARK_CODE_FRONT_IMG', 'MARK_CODE_BACK_IMG', 'ITEM_ENG_NAME', 'CHANGE_DATE', 'MARK_CODE_FRONT', 'MARK_CODE_BACK', 'EDI_CODE', 'BIZRNO', 'STD_CD']

샘플 1행:


,0
ITEM_SEQ,200808876
ITEM_NAME,가스디알정50밀리그램(디메크로틴산마그네슘)
ENTP_SEQ,19540006
ENTP_NAME,일동제약(주)
CHART,녹색의 원형 필름코팅정
ITEM_IMAGE,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...
PRINT_FRONT,IDG
PRINT_BACK,None
DRUG_SHAPE,원형
COLOR_CLASS1,연두


In [19]:
# 핵심 식별자 컬럼 존재 여부 확인
id_cols = ["ITEM_SEQ", "ITEM_NAME", "ENTP_NAME", "CHART", "DRUG_SHAPE", "COLOR_CLASS1",
           "PRINT_FRONT", "PRINT_BACK", "ITEM_IMAGE"]

if items:
    found = [c for c in id_cols if c in df_sample.columns]
    missing = [c for c in id_cols if c not in df_sample.columns]
    print("존재:", found)
    print("누락:", missing)

존재: ['ITEM_SEQ', 'ITEM_NAME', 'ENTP_NAME', 'CHART', 'DRUG_SHAPE', 'COLOR_CLASS1', 'PRINT_FRONT', 'PRINT_BACK', 'ITEM_IMAGE']
누락: []


In [20]:
import re, time
import requests
import pandas as pd
from pathlib import Path

ROOT = Path("../..").resolve()
INTERIM = ROOT / "data" / "interim"

df_nadal = pd.read_parquet(INTERIM / "nadal_ident_clean.parquet")

# ITEM_NAME 패턴: "약품명(성분명)" — 마지막 괄호 안 추출
def extract_ingr(name):
    if not isinstance(name, str):
        return None
    m = re.search(r'\(([^)]+)\)$', name.strip())
    return m.group(1) if m else None

df_nadal["INGR_KR"] = df_nadal["ITEM_NAME"].apply(extract_ingr)
hit = df_nadal["INGR_KR"].notna().mean()
print(f"ITEM_NAME에서 성분명 추출률: {hit:.1%}")
print(df_nadal[["ITEM_SEQ","ITEM_NAME","INGR_KR"]].head(5).to_string())

ITEM_NAME에서 성분명 추출률: 68.1%
    ITEM_SEQ                ITEM_NAME         INGR_KR
0  200808876  가스디알정50밀리그램(디메크로틴산마그네슘)      디메크로틴산마그네슘
1  200808877       페라트라정2.5밀리그램(레트로졸)            레트로졸
2  200808948         졸뎀속붕정(졸피뎀타르타르산염)       졸피뎀타르타르산염
3  200809076    가스프렌정(모사프리드시트르산염이수화물)  모사프리드시트르산염이수화물
4  200809361               바르탄정(발사르탄)            발사르탄


In [21]:
import re, time, json, requests
import pandas as pd
from pathlib import Path

ROOT = Path("../..").resolve()
INTERIM = ROOT / "data" / "interim"

df_nadal = pd.read_parquet(INTERIM / "nadal_ident_clean.parquet")

# ITEM_NAME에서 한국어 성분명 추출
def extract_ingr(name):
    if not isinstance(name, str): return None
    m = re.search(r'\(([^)]+)\)$', name.strip())
    return m.group(1) if m else None

# ITEM_ENG_NAME에서 영문 성분명 추출
def extract_eng_ingr(s):
    if not isinstance(s, str): return None
    m = re.search(r'\(([A-Za-z][^)]+)\)', s)
    return m.group(1).strip() if m else None

df_nadal["INGR_KR"] = df_nadal["ITEM_NAME"].apply(extract_ingr)
df_nadal["INGR_EN"] = df_nadal["ITEM_ENG_NAME"].apply(extract_eng_ingr)

# 한국어 성분명 → 영문 INN 변환 (salt 접미사 제거 방식)
KR_SALT = {
    "칼슘삼수화물": "calcium trihydrate",
    "칼슘이수화물": "calcium dihydrate",
    "나트륨이수화물": "sodium dihydrate",
    "칼슘일수화물": "calcium monohydrate",
    "브롬화수소산염": "hydrobromide",
    "시트르산염이수화물": "citrate dihydrate",
    "시트르산염수화물": "citrate hydrate",
    "시트르산염": "citrate",
    "염산염수화물": "hydrochloride hydrate",
    "염산염": "hydrochloride",
    "황산염": "sulfate",
    "옥살산염": "oxalate",
    "말레산염": "maleate",
    "푸마르산염": "fumarate",
    "타르타르산염": "tartrate",
    "베실산염": "besylate",
    "메실산염": "mesylate",
    "토실산염": "tosylate",
    "인산수소염": "phosphate",
    "인산염": "phosphate",
    "아세트산염": "acetate",
    "말산염": "malate",
    "숙신산염일수화물": "succinate monohydrate",
    "숙신산염": "succinate",
    "글루코네이트": "gluconate",
    "실렉세틸": "cilexetil",
    "피발산염": "pivoxil",
    "삼수화물": "trihydrate",
    "이수화물": "dihydrate",
    "일수화물": "monohydrate",
    "수화물": "hydrate",
    "칼슘": "calcium",
    "나트륨": "sodium",
    "칼륨": "potassium",
    "마그네슘": "magnesium",
    "아연": "zinc",
}

# 한국어 INN → 영문 INN 직접 대응 (다빈도 성분 수동 매핑)
KR_INN_MAP = {
    "아세트아미노펜": "acetaminophen",
    "이부프로펜": "ibuprofen",
    "아스피린": "aspirin",
    "메트포르민": "metformin",
    "암로디핀": "amlodipine",
    "아토르바스타틴": "atorvastatin",
    "로수바스타틴": "rosuvastatin",
    "오메프라졸": "omeprazole",
    "에소메프라졸": "esomeprazole",
    "란소프라졸": "lansoprazole",
    "판토프라졸": "pantoprazole",
    "라베프라졸": "rabeprazole",
    "로사르탄": "losartan",
    "발사르탄": "valsartan",
    "텔미사르탄": "telmisartan",
    "칸데사르탄": "candesartan",
    "올메사르탄": "olmesartan",
    "이르베사르탄": "irbesartan",
    "리시노프릴": "lisinopril",
    "에날라프릴": "enalapril",
    "메토프롤롤": "metoprolol",
    "카르베딜롤": "carvedilol",
    "비소프롤롤": "bisoprolol",
    "클로피도그렐": "clopidogrel",
    "아토르바스타틴": "atorvastatin",
    "프레가발린": "pregabalin",
    "가바펜틴": "gabapentin",
    "트라마돌": "tramadol",
    "알프라졸람": "alprazolam",
    "레보티록신": "levothyroxine",
    "푸로세미드": "furosemide",
    "스피로노락톤": "spironolactone",
    "도네페질": "donepezil",
    "메만틴": "memantine",
    "리바록사반": "rivaroxaban",
    "아픽사반": "apixaban",
    "다비가트란": "dabigatran",
    "글리메피리드": "glimepiride",
    "글리클라지드": "gliclazide",
    "시타글립틴": "sitagliptin",
    "다파글리플로진": "dapagliflozin",
    "엠파글리플로진": "empagliflozin",
    "에제티미브": "ezetimibe",
    "피나스테리드": "finasteride",
    "타다라필": "tadalafil",
    "실데나필": "sildenafil",
    "몬테루카스트": "montelukast",
    "세티리진": "cetirizine",
    "펙소페나딘": "fexofenadine",
    "클래리트로마이신": "clarithromycin",
    "아지트로마이신": "azithromycin",
    "세파클러": "cefaclor",
    "알벤다졸": "albendazole",
    "이트라코나졸": "itraconazole",
    "플루코나졸": "fluconazole",
    "세레콕시브": "celecoxib",
    "나프록센": "naproxen",
    "디클로페낙": "diclofenac",
    "인도메타신": "indomethacin",
    "멜록시캄": "meloxicam",
    "에스시탈로프람": "escitalopram",
    "플루옥세틴": "fluoxetine",
    "설트랄린": "sertraline",
    "파록세틴": "paroxetine",
    "벤라팍신": "venlafaxine",
    "미르타자핀": "mirtazapine",
    "쿠에티아핀": "quetiapine",
    "올란자핀": "olanzapine",
    "아리피프라졸": "aripiprazole",
    "리스페리돈": "risperidone",
    "모사프리드": "mosapride",
    "이토프리드": "itopride",
}

def kr_to_en(kr_ingr):
    if not isinstance(kr_ingr, str): return None
    # 직접 매핑 우선
    base = kr_ingr
    suffix_en = ""
    for kr_sfx, en_sfx in KR_SALT.items():
        if kr_ingr.endswith(kr_sfx):
            base = kr_ingr[:-len(kr_sfx)]
            suffix_en = " " + en_sfx
            break
    if base in KR_INN_MAP:
        return KR_INN_MAP[base] + suffix_en
    if kr_ingr in KR_INN_MAP:
        return KR_INN_MAP[kr_ingr]
    return None

df_nadal["INGR_EN_DERIVED"] = df_nadal["INGR_KR"].apply(kr_to_en)
# 최종 영문 성분명: INGR_EN 우선, 없으면 파생
df_nadal["INGR_EN_FINAL"] = df_nadal["INGR_EN"].combine_first(df_nadal["INGR_EN_DERIVED"])

covered = df_nadal["INGR_EN_FINAL"].notna().mean()
print(f"영문 성분명 커버리지: {covered:.1%} ({df_nadal['INGR_EN_FINAL'].notna().sum():,}/{len(df_nadal):,})")
print(f"  - ITEM_ENG_NAME 추출: {df_nadal['INGR_EN'].notna().sum():,}건")
print(f"  - KR→EN 파생: {df_nadal['INGR_EN_DERIVED'].notna().sum():,}건")

영문 성분명 커버리지: 30.5% (7,783/25,518)
  - ITEM_ENG_NAME 추출: 3,375건
  - KR→EN 파생: 5,414건


In [22]:
# 영문 성분명으로 PubChem CID 조회
def get_pubchem_cid(name, retries=2):
    if not isinstance(name, str): return None
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(name)}/cids/JSON"
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                return r.json()["IdentifierList"]["CID"][0]
            elif r.status_code == 404:
                return None
        except Exception:
            pass
        time.sleep(0.5)
    return None

cache_path = INTERIM / "ingr_cid_cache.json"
crosswalk = {}
if cache_path.exists():
    with open(cache_path, encoding="utf-8") as f:
        crosswalk = json.load(f)

# 유일 영문 성분명만 조회
unique_en = df_nadal["INGR_EN_FINAL"].dropna().unique().tolist()
print(f"조회 대상 유일 성분명: {len(unique_en)}종")

new_queries = [n for n in unique_en if n not in crosswalk]
print(f"신규 조회: {len(new_queries)}종")

for name in new_queries:
    crosswalk[name] = get_pubchem_cid(name)
    time.sleep(0.12)

with open(cache_path, "w", encoding="utf-8") as f:
    json.dump(crosswalk, f, ensure_ascii=False)

hits = sum(1 for v in crosswalk.values() if v is not None)
print(f"CID 조회 성공: {hits}/{len(crosswalk)} ({hits/max(len(crosswalk),1):.1%})")

조회 대상 유일 성분명: 1216종
신규 조회: 1216종
CID 조회 성공: 793/1216 (65.2%)


In [23]:
# crosswalk 테이블 생성 및 저장
df_cross = df_nadal[["ITEM_SEQ", "ITEM_NAME", "INGR_KR", "INGR_EN_FINAL"]].copy()
df_cross["PUBCHEM_CID"] = df_cross["INGR_EN_FINAL"].map(crosswalk)
df_cross["TWOSIDES_ID"] = df_cross["PUBCHEM_CID"].apply(
    lambda x: f"CID{int(x):09d}" if pd.notna(x) else None
)

coverage = df_cross["TWOSIDES_ID"].notna().mean()
print(f"품목일련번호 → TWOSIDES_ID 커버리지: {coverage:.1%} ({df_cross['TWOSIDES_ID'].notna().sum():,}/{len(df_cross):,})")

df_cross.to_parquet(INTERIM / "korean_cid_crosswalk.parquet", index=False)
print("저장: data/interim/korean_cid_crosswalk.parquet")
df_cross[df_cross["TWOSIDES_ID"].notna()].head(5)

품목일련번호 → TWOSIDES_ID 커버리지: 26.3% (6,702/25,518)
저장: data/interim/korean_cid_crosswalk.parquet


,ITEM_SEQ,ITEM_NAME,INGR_KR,INGR_EN_FINAL,PUBCHEM_CID,TWOSIDES_ID
3,200809076,가스프렌정(모사프리드시트르산염이수화물),모사프리드시트르산염이수화물,mosapride citrate dihydrate,6918047.0,CID006918047
4,200809361,바르탄정(발사르탄),발사르탄,valsartan,60846.0,CID000060846
5,200809402,리피논정80밀리그램(아토르바스타틴칼슘삼수화물),아토르바스타틴칼슘삼수화물,Atorvastatin calcium trihydrate,656846.0,CID000656846
6,200809710,사르발탄정160밀리그램(발사르탄),발사르탄,valsartan,60846.0,CID000060846
7,200809883,아푸르탄정150밀리그램(이르베사르탄),이르베사르탄,irbesartan,3749.0,CID000003749
